In [1]:
!git clone https://github.com/DiMarzioBian/ABXI.git 

Cloning into 'ABXI'...


In [3]:
domains = ['afk', 'abe', 'amb'] 

domain_names = {
    'afk': ('Food', 'Kitchen'),
    'abe': ('Beauty', 'Electronic'),
    'amb': ('Movie', 'Book')
}

# Helper Functions

In [18]:
import polars as pl
import pickle as pkl
import numpy as np
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

def read_sequence_file(path: str) -> pd.DataFrame:
    """
    Reads a sequence file where each line is:
      uid item|timestamp item|timestamp ...

    Returns a DataFrame with columns:
      (uid, item_id, timestamp)
    """
    rows = []

    with open(path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 2:
                continue

            uid = int(parts[0])

            for token in parts[1:]:
                item_id, ts = token.split('|')
                rows.append((uid, int(item_id), int(ts)))

    return pd.DataFrame(rows, columns=['uid', 'item_id', 'timestamp'])

def leave_one_out(
        data,
        key = 'userid',
        target = None,
        sample_top = False,
        random_state = None
    ):
    if sample_top: # sample item with the highest target value (e.g., rating, time, etc.)
        idx = (
            data[target]
            .sample(frac=1, random_state=random_state) # handle same feedback for different items
            .groupby(data[key], sort=False)
            .idxmax()
        ).values
    else: # sample random item
        idx = (
            data[key]
            .sample(frac=1, random_state=random_state)
            .drop_duplicates(keep='first') # data is shuffled - simply take the 1st element
            .index
        ).values
    observed = data.drop(idx)
    holdout = data.loc[idx]
    return observed, holdout



In [19]:
def parse_domain_data(domain):
    data = pkl.load(open(f'./ABXI/data/{domain}/{domain}_50_seq.pkl', 'rb'))
    n_A_items = data[3]
    n_users = len(data[0])
    gt_val_items = np.array([data[1][i][3] for i in range(n_users)]).reshape(-1)
    gt_test_items = np.array([data[2][i][3] for i in range(n_users)]).reshape(-1)
    df = read_sequence_file(f'./ABXI/data/{domain}/{domain}_50_preprocessed.txt')
    df = df.sort_values(['uid', 'timestamp'])
    train, test = leave_one_out(df, 'uid', 'timestamp', sample_top=True)
    train, val = leave_one_out(train, 'uid', 'timestamp', sample_top=True)
    test = test.sort_values('uid')
    val = val.sort_values('uid')
    val['item_id'] = gt_val_items
    test['item_id'] = gt_test_items

    return df, train, val, test, n_A_items

In [20]:

def save(domain, df, train, val, test, n_A_items):
    print(f'Processing domain: {domain}: {domain_names[domain]}')
    domain_A_name, domain_B_name = domain_names[domain]
    path = os.path.join('./data', domain)
    os.makedirs(path, exist_ok=True)
    os.makedirs(path + f'/{domain_A_name}', exist_ok=True)
    os.makedirs(path + f'/{domain_B_name}', exist_ok=True)
    print(domain.upper(), domain[1].upper(), domain[2].upper(), sep='\t')
    for name, d in [('all', df), ('train', train), ('val',val) , ('test',test)]:
        dA = d.query('item_id <= @n_A_items')
        dB = d.query('item_id > @n_A_items')
        dB['item_id'] = dB['item_id'] - n_A_items
        print(name, len(dA), len(dB), sep='\t')
        if name!='all':
            dA.to_parquet(path + f'/{domain_A_name}/{name}.parquet', index=False)
            dB.to_parquet(path + f'/{domain_B_name}/{name}.parquet', index=False)
        else:
            A_items = dA['item_id'].nunique()
            B_items = dB['item_id'].nunique()
            pkl.dump({k:k for k in range(dA['item_id'].nunique())}, open(path + f'/{domain_A_name}/item_id_to_idx.pkl', 'wb'))
            pkl.dump({k:k for k in range(dB['item_id'].nunique())}, open(path + f'/{domain_B_name}/item_id_to_idx.pkl', 'wb'))
            print('items', A_items, B_items, sep='\t')

    

# Process ABXI datasets

Run the preprocessing code to check the authenticity of the preprocessing pipeline

![alt text](../images/ABXI_data_stats.png)

In [21]:
for domain in domains:
    df, train, val, test, n_A_items = parse_domain_data(domain)
    save(domain, df, train, val, test, n_A_items)

Processing domain: afk: ('Food', 'Kitchen')
AFK	F	K
all	83663	89885
items	11837	16258
train	77473	81787
val	2837	4307
test	2419	4725
Processing domain: abe: ('Beauty', 'Electronic')
ABE	B	E
all	50329	63800
items	10379	14188
train	45823	59358
val	2086	2388
test	1875	2599
Processing domain: amb: ('Movie', 'Book')
AMB	M	B
all	347654	403147
items	35712	90958
train	321767	372334
val	11728	16622
test	10935	17415


In [ ]:
!rm -rf ABXI